# 🏪 Sistema de Conteo de Clientes en Tienda - Visión por Computadora

Este notebook empaqueta todos los módulos del proyecto actual para la estabilización, marcado, procesamiento y conteo de clientes en un quiosco/tienda comercial.

### 📌 Módulos incluidos:
1. **Instalación y Configuración del Entorno**: Soporte para ejecución local y en Google Colab.
2. **Estabilización y Recorte ROI (`estabilizar.py`)**: Alineación por flujo óptico (Lucas-Kanade) y recorte del área útil del quiosco (`estabilizado_roi.mp4`).
3. **Marcado y Definición de Zonas (`marcar.py`)**: Gestión de zonas relativas/normalizadas guardadas en `espacios.pkl`.
4. **Método 1: Visión Tradicional con MOG2 (`contar.py`)**: Sustracción de fondo adaptativa MOG2, filtrado morfológico, aumento de brillo y conteo temporal anti-transición.
5. **Método 2: Deep Learning con YOLOv11 (`contar_yolo.py`)**: Tracking multi-objeto con ByteTrack, evaluación de proximidad física a las zonas y conteo por persistencia de IDs.
6. **Comparativa y Visualización**: Gráficas y reproductor web integrado para ver los resultados en video.


In [ ]:
# ==============================================================================
# 0. DEPENDENCIAS Y ENTORNO
# ==============================================================================
import sys
import os

try:
    import google.colab
    IN_COLAB = True
    print("🚀 Detectado entorno: Google Colab")
    !pip install -q ultralytics opencv-python-headless matplotlib tqdm
except ImportError:
    IN_COLAB = False
    print("💻 Detectado entorno: Local / Jupyter")

import cv2
import pickle
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from IPython.display import HTML, display
from base64 import b64encode

print("✅ Dependencias cargadas correctamente.")


## 📁 1. Archivos y Zonas Relativas (`espacios.pkl`)

El proyecto utiliza coordenadas relativas (normalizadas entre 0 y 1) para que funcionen con cualquier resolución de video.
Si no existe `espacios.pkl`, se inicializa automáticamente con la configuración calibrada de las 3 zonas del mostrador.


In [ ]:
# ==============================================================================
# 1. VERIFICACIÓN DE VIDEO Y CONFIGURACIÓN DE ZONAS
# ==============================================================================

# Videos disponibles
VIDEO_ORIGINAL = 'video.mp4'
VIDEO_ROI = 'estabilizado_roi.mp4' if os.path.exists('estabilizado_roi.mp4') else ('estabilizado.mp4' if os.path.exists('estabilizado.mp4') else VIDEO_ORIGINAL)

print(f"🎬 Video de entrada para estabilizar: {VIDEO_ORIGINAL}")
print(f"🎬 Video enfocado para procesamiento: {VIDEO_ROI}")

# Zonas relativas calibradas por defecto (rx, ry, rw, rh)
ZONAS_RELATIVAS_DEFAULT = [
    (0.15, 0.35, 0.20, 0.65),  # Zona 1: Mostrador izquierdo
    (0.40, 0.35, 0.20, 0.65),  # Zona 2: Mostrador central
    (0.65, 0.35, 0.20, 0.65)   # Zona 3: Vitrina derecha
]

PKL_FILE = 'espacios.pkl'
if os.path.exists(PKL_FILE):
    with open(PKL_FILE, 'rb') as f:
        ZONAS_ACTUALES = pickle.load(f)
    print(f"📦 '{PKL_FILE}' cargado ({len(ZONAS_ACTUALES)} zonas).")
else:
    with open(PKL_FILE, 'wb') as f:
        pickle.dump(ZONAS_RELATIVAS_DEFAULT, f)
    ZONAS_ACTUALES = ZONAS_RELATIVAS_DEFAULT
    print(f"📦 Se generó '{PKL_FILE}' con las zonas por defecto.")

print(f"📍 Zonas relativas configuradas: {ZONAS_ACTUALES}")


## 🔄 2. Módulo de Estabilización y Recorte ROI (`estabilizar.py`)

Compensa vibraciones o movimientos de cámara mediante flujo óptico de Lucas-Kanade sobre la estructura rígida del edificio, extrayendo y recortando exclusivamente la región de la tienda (`ROI_TOP_RATIO = 0.42`, `ROI_BOTTOM_RATIO = 0.74`).


In [ ]:
# ==============================================================================
# 2. ESTABILIZACIÓN Y RECORTE DE LA REGIÓN DE INTERÉS (estabilizar.py)
# ==============================================================================

def ejecutar_estabilizacion_roi(
    video_source='video.mp4',
    output_video='estabilizado_roi.mp4',
    reference_second=293,
    top_ratio=0.42,
    bottom_ratio=0.74,
    left_ratio=0.10,
    right_ratio=0.90
):
    if not os.path.exists(video_source):
        print(f"⚠️ No se encontró '{video_source}'. Omitiendo paso de estabilización.")
        return False
        
    capture = cv2.VideoCapture(video_source)
    fps = capture.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    
    capture.set(cv2.CAP_PROP_POS_MSEC, reference_second * 1000)
    success, reference_frame = capture.read()
    if not success:
        capture.set(cv2.CAP_PROP_POS_FRAMES, 0)
        success, reference_frame = capture.read()
        
    capture.set(cv2.CAP_PROP_POS_FRAMES, 0)
    full_height, full_width = reference_frame.shape[:2]
    
    roi_y_start = int(full_height * top_ratio)
    roi_y_end = int(full_height * bottom_ratio)
    roi_x_start = int(full_width * left_ratio)
    roi_x_end = int(full_width * right_ratio)
    
    output_width = roi_x_end - roi_x_start
    output_height = roi_y_end - roi_y_start
    
    writer = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (output_width, output_height))
    
    reference_gray = cv2.cvtColor(reference_frame, cv2.COLOR_BGR2GRAY)
    building_mask = np.zeros((full_height, full_width), dtype=np.uint8)
    building_mask[roi_y_start:roi_y_end, roi_x_start:roi_x_end] = 255
    
    reference_points = cv2.goodFeaturesToTrack(
        reference_gray, maxCorners=300, qualityLevel=0.01, minDistance=15, mask=building_mask
    )
    
    print(f"🎬 Estabilizando y recortando: {total_frames} fotogramas...")
    pbar = tqdm(total=total_frames, desc="Estabilizando ROI")
    
    while True:
        success, current_frame = capture.read()
        if not success:
            break
            
        current_gray = cv2.cvtColor(current_frame, cv2.COLOR_BGR2GRAY)
        tracked_points, status, _ = cv2.calcOpticalFlowPyrLK(
            reference_gray, current_gray, reference_points, None, winSize=(21, 21), maxLevel=3
        )
        
        valid_ref = reference_points[status == 1]
        valid_cur = tracked_points[status == 1]
        
        if len(valid_cur) >= 10:
            affine_matrix, _ = cv2.estimateAffinePartial2D(valid_cur, valid_ref)
            if affine_matrix is not None:
                stabilized = cv2.warpAffine(current_frame, affine_matrix, (full_width, full_height))
            else:
                stabilized = current_frame
        else:
            stabilized = current_frame
            
        roi_frame = stabilized[roi_y_start:roi_y_end, roi_x_start:roi_x_end]
        writer.write(roi_frame)
        pbar.update(1)
        
    pbar.close()
    capture.release()
    writer.release()
    print(f"✅ Video estabilizado y recortado guardado en: {output_video}")
    return True

# Descomentar para ejecutar si se desea re-estabilizar:
# ejecutar_estabilizacion_roi('video.mp4', 'estabilizado_roi.mp4')


## 📐 3. Visualización del Primer Fotograma y Zonas de Interés

Comprueba visualmente las 3 zonas del mostrador superpuestas sobre el video recortado.


In [ ]:
# ==============================================================================
# 3. VISUALIZACIÓN DE ZONAS RELATIVAS SOBRE EL VIDEO
# ==============================================================================
cap = cv2.VideoCapture(VIDEO_ROI)
ok, frame = cap.read()
cap.release()

if ok:
    h, w = frame.shape[:2]
    frame_draw = frame.copy()
    
    with open('espacios.pkl', 'rb') as f:
        zones = pickle.load(f)
        
    for i, (rx, ry, rw, rh) in enumerate(zones):
        px = int(rx * w) if rx <= 1.0 else int(rx)
        py = int(ry * h) if ry <= 1.0 else int(ry)
        pw = int(rw * w) if rw <= 1.0 else int(rw)
        ph = int(rh * h) if rh <= 1.0 else int(rh)
        
        cv2.rectangle(frame_draw, (px, py), (px + pw, py + ph), (0, 255, 0), 2)
        cv2.putText(frame_draw, f"Zona {i+1}", (px + 6, py + 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(frame_draw, cv2.COLOR_BGR2RGB))
    plt.title(f"Zonas de Mostrador ({w}x{h} px) - {VIDEO_ROI}", fontsize=13)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠️ No se pudo leer el video: {VIDEO_ROI}")


## 👥 4. Método 1: Visión Tradicional con Sustracción MOG2 (`contar.py`)

Implementa la versión actual de [contar.py](contar.py):
- **Aumento de Brillo**: `current_frame = cv2.convertScaleAbs(current_frame, beta=BRIGHTNESS_OFFSET)`.
- **Sustracción de Fondo Adaptativa**: `cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=25)`.
- **Morfología Matemática**: Operaciones `MORPH_OPEN` y `MORPH_CLOSE` para eliminar sombras suaves y sellar siluetas de personas.
- **Filtro Anti-Transición entre Zonas**: Si una persona ya fue contada y se desplaza lateralmente a una zona vecina, hereda el estado sin duplicar el conteo.


In [ ]:
# ==============================================================================
# 4. EJECUCIÓN MÉTODO TRADICIONAL (contar.py)
# ==============================================================================

def procesar_vision_tradicional(
    video_path=VIDEO_ROI,
    output_path='resultado_tradicional.mp4',
    brightness_offset=30,
    detection_threshold=0.08,
    min_seconds_occupied=1.3,
    min_seconds_free=1.0,
    vertical_shift_ratio=0.06,
    max_frames=None
):
    capture = cv2.VideoCapture(video_path)
    fps = capture.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total_frames = min(total_frames, max_frames)
        
    success, sample_frame = capture.read()
    if not success:
        print("❌ Error: No se pudo abrir el archivo de video.")
        return [], []
        
    frame_height, frame_width = sample_frame.shape[:2]
    capture.set(cv2.CAP_PROP_POS_FRAMES, 0)
    
    vertical_offset_pixels = int(frame_height * vertical_shift_ratio)
    
    # Cargar y convertir zonas relativas a píxeles
    pixel_zones = []
    try:
        with open('espacios.pkl', 'rb') as file:
            raw_zones = pickle.load(file)
        for x, y, w, h in raw_zones:
            zx = int(x * frame_width) if x <= 1.0 else int(x)
            zy = (int(y * frame_height) if y <= 1.0 else int(y)) + vertical_offset_pixels
            zw = int(w * frame_width) if w <= 1.0 else int(w)
            zh = int(h * frame_height) if h <= 1.0 else int(h)
            if zy + zh > frame_height:
                zh = frame_height - zy
            if 0 <= zx < frame_width and 0 <= zy < frame_height:
                pixel_zones.append((zx, zy, zw, zh))
    except Exception:
        pixel_zones = []

    if len(pixel_zones) != 3:
        box_start_y = int(frame_height * 0.35)
        box_height = frame_height - box_start_y
        box_width = int(frame_width * 0.20)
        pixel_zones = [
            (int(frame_width * 0.15), box_start_y, box_width, box_height),
            (int(frame_width * 0.40), box_start_y, box_width, box_height),
            (int(frame_width * 0.65), box_start_y, box_width, box_height)
        ]
        
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))
    
    background_subtractor = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=25, detectShadows=True)
    morphology_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    
    occupied_frames = [0] * len(pixel_zones)
    free_frames = [0] * len(pixel_zones)
    zone_customer_active = [False] * len(pixel_zones)
    client_counters = [0] * len(pixel_zones)
    events = []
    
    print(f"🎬 Procesando {total_frames} fotogramas con MOG2...")
    pbar = tqdm(total=total_frames, desc="Visión Clásica")
    frame_index = 0
    
    while True:
        success, current_frame = capture.read()
        if not success or (max_frames and frame_index >= max_frames):
            break
            
        current_frame = cv2.convertScaleAbs(current_frame, beta=brightness_offset)
        frame_index += 1
        
        foreground_mask = background_subtractor.apply(current_frame, learningRate=0.001)
        _, clean_binary_mask = cv2.threshold(foreground_mask, 200, 255, cv2.THRESH_BINARY)
        denoised_mask = cv2.morphologyEx(clean_binary_mask, cv2.MORPH_OPEN, morphology_kernel)
        solid_mask = cv2.morphologyEx(denoised_mask, cv2.MORPH_CLOSE, morphology_kernel)
        
        for index, (x, y, width, height) in enumerate(pixel_zones):
            zone_region = solid_mask[y:y + height, x:x + width]
            non_zero_pixels = cv2.countNonZero(zone_region)
            occupancy_ratio = non_zero_pixels / max(1, (width * height))
            
            if occupancy_ratio > detection_threshold:
                occupied_frames[index] += 1
                free_frames[index] = 0
                
                neighbor_already_active = False
                for neighbor_index in [index - 1, index + 1]:
                    if 0 <= neighbor_index < len(pixel_zones):
                        if zone_customer_active[neighbor_index]:
                            neighbor_already_active = True
                            break
                            
                if occupied_frames[index] >= int(min_seconds_occupied * fps) and not zone_customer_active[index]:
                    if not neighbor_already_active:
                        client_counters[index] += 1
                        zone_customer_active[index] = True
                        events.append((frame_index / fps, index + 1, client_counters[index]))
                    else:
                        zone_customer_active[index] = True
            else:
                free_frames[index] += 1
                if free_frames[index] >= int(min_seconds_free * fps):
                    occupied_frames[index] = 0
                    zone_customer_active[index] = False
                    
            if zone_customer_active[index]:
                box_color = (0, 0, 255)
            elif occupied_frames[index] > 0:
                box_color = (0, 255, 255)
            else:
                box_color = (0, 255, 0)
                
            cv2.rectangle(current_frame, (x, y), (x + width, y + height), box_color, 2)
            cv2.putText(
                current_frame,
                f"Z{index + 1}: {client_counters[index]} ({occupancy_ratio * 100:.0f}%)",
                (x, y - 6),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                box_color,
                2
            )
            
        cv2.putText(current_frame, f"TOTAL: {sum(client_counters)}", (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
        writer.write(current_frame)
        pbar.update(1)
        
    pbar.close()
    capture.release()
    writer.release()
    
    print("
" + "="*50)
    print("📊 RESULTADOS VISIÓN TRADICIONAL:")
    for i, count in enumerate(client_counters):
        print(f"   • Zona {i + 1}: {count} clientes")
    print(f"   ⭐ TOTAL CLIENTES: {sum(client_counters)}")
    print(f"   📁 Video generado: {output_path}")
    print("="*50)
    
    return client_counters, events

# Ejecutar:
# conteo_trad, eventos_trad = procesar_vision_tradicional(VIDEO_ROI, 'resultado_tradicional.mp4')


## 📺 Reproductor de Video en Celda
Permite visualizar videos procesados directamente dentro de Google Colab o Jupyter convirtiendo a H.264 compatible.


In [ ]:
# ==============================================================================
# FUNCIÓN DE REPRODUCCIÓN WEB EN EL NOTEBOOK
# ==============================================================================
def mostrar_video_notebook(video_file, ancho=600):
    web_file = "web_" + video_file
    os.system(f"ffmpeg -y -i {video_file} -vcodec libx264 -crf 23 -pix_fmt yuv420p -acodec aac {web_file} >/dev/null 2>&1")
    target = web_file if os.path.exists(web_file) else video_file
    
    video_bytes = open(target, 'rb').read()
    video_b64 = b64encode(video_bytes).decode()
    
    return HTML(f'''
    <div style="text-align:center;">
        <video width="{ancho}" controls autoplay loop style="border-radius:8px; box-shadow: 0 4px 10px rgba(0,0,0,0.3);">
            <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
            Tu navegador no soporta reproducción HTML5 directa.
        </video>
    </div>
    ''')

# Ejemplo de uso:
# mostrar_video_notebook('resultado_tradicional.mp4', ancho=650)


## 🤖 5. Método 2: Deep Learning con YOLOv11 (`contar_yolo.py`)

Implementa la versión actual de [contar_yolo.py](contar_yolo.py):
- **Modelo**: `yolo11n.pt` para detección de personas (`classes=[0]`).
- **Punto de Referencia Corporal**: Se proyecta al 85% de la altura de la persona (`y1 + int((y2 - y1) * 0.85)`), alineándose con la base de atención del mostrador.
- **Tracking Multi-Objeto**: Cada cliente recibe un `ID` numérico. Solo se cuenta un cliente cuando su ID permanece más de `MINIMUM_SECONDS_OCCUPIED = 1.3s` en la zona.


In [ ]:
# ==============================================================================
# 5. EJECUCIÓN MÉTODO YOLOv11 (contar_yolo.py)
# ==============================================================================
from ultralytics import YOLO

def procesar_conteo_yolo(
    video_path=VIDEO_ROI,
    output_path='resultado_yolo.mp4',
    model_name='yolo11n.pt',
    min_seconds_occupied=1.3,
    frame_step=2,
    analysis_size=640,
    vertical_shift_ratio=0.06,
    max_frames=None
):
    print(f"📦 Cargando modelo YOLO ({model_name})...")
    model = YOLO(model_name)
    
    capture = cv2.VideoCapture(video_path)
    fps = capture.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total_frames = min(total_frames, max_frames)
        
    success, sample_frame = capture.read()
    if not success:
        print("❌ Error: No se pudo abrir el archivo de video.")
        return [], []
        
    frame_height, frame_width = sample_frame.shape[:2]
    capture.set(cv2.CAP_PROP_POS_FRAMES, 0)
    
    vertical_offset_pixels = int(frame_height * vertical_shift_ratio)
    
    pixel_zones = []
    try:
        with open('espacios.pkl', 'rb') as file:
            raw_zones = pickle.load(file)
        for x, y, w, h in raw_zones:
            zx = int(x * frame_width) if x <= 1.0 else int(x)
            zy = (int(y * frame_height) if y <= 1.0 else int(y)) + vertical_offset_pixels
            zw = int(w * frame_width) if w <= 1.0 else int(w)
            zh = int(h * frame_height) if h <= 1.0 else int(h)
            if zy + zh > frame_height:
                zh = frame_height - zy
            if 0 <= zx < frame_width and 0 <= zy < frame_height:
                pixel_zones.append((zx, zy, zw, zh))
    except Exception:
        pixel_zones = []

    if len(pixel_zones) != 3:
        box_start_y = int(frame_height * 0.35)
        box_height = frame_height - box_start_y
        box_width = int(frame_width * 0.20)
        pixel_zones = [
            (int(frame_width * 0.15), box_start_y, box_width, box_height),
            (int(frame_width * 0.40), box_start_y, box_width, box_height),
            (int(frame_width * 0.65), box_start_y, box_width, box_height)
        ]
        
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))
    
    person_dwell_frames = {}
    counted_customer_ids = set()
    client_counters = [0] * len(pixel_zones)
    events = []
    
    print(f"🎬 Procesando {total_frames} fotogramas con YOLOv11...")
    pbar = tqdm(total=total_frames, desc="YOLOv11 Tracking")
    frame_index = 0
    
    while True:
        success, current_frame = capture.read()
        if not success or (max_frames and frame_index >= max_frames):
            break
            
        frame_index += 1
        if frame_index % frame_step != 0:
            continue
            
        tracking_results = model.track(
            current_frame,
            persist=True,
            classes=[0],
            imgsz=analysis_size,
            verbose=False
        )[0]
        
        active_ids_per_zone = [[] for _ in range(len(pixel_zones))]
        
        if tracking_results.boxes.id is not None:
            bounding_boxes = tracking_results.boxes.xyxy.int().tolist()
            tracking_ids = tracking_results.boxes.id.int().tolist()
            
            for (x1, y1, x2, y2), person_id in zip(bounding_boxes, tracking_ids):
                reference_point_x = (x1 + x2) // 2
                reference_point_y = y1 + int((y2 - y1) * 0.85)
                
                for zone_index, (zx, zy, zw, zh) in enumerate(pixel_zones):
                    centroid_inside = (zx <= reference_point_x <= (zx + zw) and zy <= reference_point_y <= (zy + zh))
                    box_overlap = not (x2 < zx or x1 > (zx + zw) or y2 < zy or y1 > (zy + zh))
                    if centroid_inside or box_overlap:
                        active_ids_per_zone[zone_index].append(person_id)
                        
                box_color = (0, 0, 255) if person_id in counted_customer_ids else (255, 0, 255)
                cv2.rectangle(current_frame, (x1, y1), (x2, y2), box_color, 2)
                cv2.putText(current_frame, f"ID {person_id}", (x1, y1 - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)
                cv2.circle(current_frame, (reference_point_x, reference_point_y), 4, (0, 255, 255), -1)
                
        for index, (x, y, width, height) in enumerate(pixel_zones):
            zone_ids = active_ids_per_zone[index]
            zone_has_counted = False
            zone_has_pending = False
            
            for person_id in zone_ids:
                person_dwell_frames[person_id] = person_dwell_frames.get(person_id, 0) + frame_step
                if person_dwell_frames[person_id] >= int(min_seconds_occupied * fps):
                    if person_id not in counted_customer_ids:
                        counted_customer_ids.add(person_id)
                        client_counters[index] += 1
                        events.append((frame_index / fps, index + 1, person_id))
                        
                if person_id in counted_customer_ids:
                    zone_has_counted = True
                else:
                    zone_has_pending = True
                    
            if zone_has_counted:
                zone_color = (0, 0, 255)
            elif zone_has_pending:
                zone_color = (0, 255, 255)
            else:
                zone_color = (0, 255, 0)
                
            cv2.rectangle(current_frame, (x, y), (x + width, y + height), zone_color, 2)
            cv2.putText(current_frame, f"Z{index + 1}: {client_counters[index]}", (x + 4, y + 22), cv2.FONT_HERSHEY_SIMPLEX, 0.6, zone_color, 2)
            
        cv2.putText(current_frame, f"TOTAL: {len(counted_customer_ids)}", (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
        writer.write(current_frame)
        pbar.update(1)
        
    pbar.close()
    capture.release()
    writer.release()
    
    print("
" + "="*50)
    print("📊 RESULTADOS YOLOv11:")
    for i, count in enumerate(client_counters):
        print(f"   • Zona {i + 1}: {count} clientes")
    print(f"   ⭐ TOTAL CLIENTES: {len(counted_customer_ids)}")
    print(f"   📁 Video generado: {output_path}")
    print("="*50)
    
    return client_counters, events

# Ejecutar:
# conteo_yolo, eventos_yolo = procesar_conteo_yolo(VIDEO_ROI, 'resultado_yolo.mp4')


## 📊 6. Comparativa de Resultados entre Ambos Métodos

Permite graficar y contrastar la cantidad de clientes detectados por zona entre el método tradicional MOG2 y YOLOv11.


In [ ]:
# ==============================================================================
# 6. GRÁFICA COMPARATIVA DE CLIENTES POR ZONA
# ==============================================================================
# Valores de prueba / ejecución
resultados_trad = [1, 2, 2]  # Actualizar tras ejecutar procesar_vision_tradicional
resultados_yolo = [1, 2, 2]  # Actualizar tras ejecutar procesar_conteo_yolo

zonas_labels = [f"Zona {i + 1}" for i in range(len(resultados_trad))]
x = np.arange(len(zonas_labels))
ancho_barra = 0.35

plt.figure(figsize=(9, 5))
plt.bar(x - ancho_barra/2, resultados_trad, width=ancho_barra, label='Visión Tradicional (MOG2)', color='#2b5c8f')
plt.bar(x + ancho_barra/2, resultados_yolo, width=ancho_barra, label='Deep Learning (YOLOv11)', color='#e05d44')

plt.xlabel('Zonas del Mostrador', fontsize=12)
plt.ylabel('Clientes Contados', fontsize=12)
plt.title('Comparativa de Clientes Atendidos por Zona', fontsize=14, fontweight='bold')
plt.xticks(x, zonas_labels)
plt.legend(fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.7)

for i in range(len(zonas_labels)):
    plt.text(x[i] - ancho_barra/2, resultados_trad[i] + 0.05, str(resultados_trad[i]), ha='center', fontweight='bold')
    plt.text(x[i] + ancho_barra/2, resultados_yolo[i] + 0.05, str(resultados_yolo[i]), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Total Visión Tradicional: {sum(resultados_trad)} clientes")
print(f"Total YOLOv11:           {sum(resultados_yolo)} clientes")
